In [ ]:
from math import *
from build123d import *
from cad_utils import *

set_port(3939)
set_viewer_config(port=3939)

In [ ]:
width = 22
height = 9.5
depth = 11
divot_depth = 3.5
grove_count = 7
grove_fillet = 0.2
grove_depth = 1.5
center_channel_width = 1.25
center_insert_depth = 1
center_insert_wedge_size = (0.75, 1, 0.5)
center_insert_tolerance = 0.15
tip_fillet = 1
outer_fillet = 0.25
bottom_chamfer = 1
lever_width = 3.5  # Neck width of DP lever
lever_thickness = 1.2  # Thickness of lever neck
lever_slot_depth = 4.5  # How deep the slot goes into the knob
lever_chamfer = 0.25  # Chamfer on the edges of the lever slot to make it easier to insert the lever
lever_tolerance = 0.1

reset_show()

with BuildPart() as slider_knob_body:
  with BuildSketch(Plane.XZ) as s:
    hw = width / 2
    hw16 = hw * (1 / 6)
    hw56 = hw * (5 / 6)
    with BuildLine():
      # Bottom line
      b = Line((-hw56, 0), (hw56, 0))
      # 45 degree lines
      rl = Line(b @ 1, (hw, hw16))
      ll = Line(b @ 0, (-hw, hw16))
      # Side arcs
      ra = RadiusArc((hw56, height), rl @ 1, radius=height)
      la = RadiusArc(ll @ 1, (-hw56, height), radius=height)
      # Divot arc
      SagittaArc(ra @ 0, la @ 1, divot_depth)
    make_face()
    divot_arc = s.edges()[3]
    with Locations(
      [
        Location(divot_arc.position_at(p), (0, 0, divot_arc.tangent_angle_at(p)))
        for p in [(i + 1) / (grove_count + 1) for i in range(grove_count) if i != grove_count // 2]
      ]
    ):
      Circle(grove_depth / 2, mode=Mode.SUBTRACT)

    tip_vertices = vertices().filter_by(lambda v: isclose(v.Y, height))
    grove_vertices = (
      vertices()
      .filter_by(lambda v: v not in tip_vertices)
      .filter_by_position(Axis.Y, height - divot_depth, height)
    )
    fillet(grove_vertices, grove_fillet)
    fillet(tip_vertices, tip_fillet)

  extrude(amount=depth / 2, both=True)

  chamfer(edges().group_by(Axis.Z)[0].filter_by(Axis.X), bottom_chamfer)

  def knob_insert_sketch(*, cut=False):
    center_plane = Plane.ZY.offset(
      -(center_channel_width + (center_insert_tolerance if cut else 0)) / 2
    )
    center = slider_knob_body.part.intersect(center_plane).face()
    with BuildSketch(center_plane, mode=Mode.PRIVATE) as s:
      p = project(center, mode=Mode.PRIVATE)
      top = p.edges().sort_by(Axis.X).last
      sides = [
        e.trim_to_length(0, e.length - (0 if cut else center_insert_tolerance))
        for e in p.edges().filter_by(Axis.X)
      ]
      o = offset(
        sides + [top],
        amount=center_insert_depth + (center_insert_tolerance if cut else 0),
        side=Side.LEFT,
        mode=Mode.ADD,
      )
      make_face()
      corners = o.vertices().sort_by_distance(Vector())[0:2]
      locs = [
        Plane(
          a,
          x_dir=Vector(b - a).normalized(),
          y_dir=(1, 0, 0),
        )
        for a, b in zip(corners, reversed(corners))
      ]
      with Locations(locs):
        if cut:
          Rectangle(
            center_insert_wedge_size[0] + center_insert_tolerance,
            center_insert_wedge_size[1] + center_insert_tolerance,
            align=Align.MIN,
          )
        else:
          with BuildSketch(center_plane, mode=Mode.PRIVATE) as knob_insert_wedge_sketch:
            a, b, c = center_insert_wedge_size
            with BuildLine() as l:
              Polyline(
                (0, 0),
                (a, 0),
                (a, c),
                (0, b),
                close=True,
              )
            make_face()
          add(knob_insert_wedge_sketch, mode=Mode.ADD)
      if not cut:
        fillet(vertices().sort_by_distance(s.sketch_local.center())[-4:], outer_fillet)
    return s.sketch

  with BuildPart(mode=Mode.PRIVATE) as slider_knob_insert:
    extrude(knob_insert_sketch(), amount=center_channel_width)

  extrude(
    knob_insert_sketch(cut=True),
    amount=center_channel_width + center_insert_tolerance,
    mode=Mode.SUBTRACT,
  )

  fillet(
    edges().filter_by(Plane.XZ).filter_by(lambda e: e.geom_type != GeomType.LINE), outer_fillet
  )

  with BuildSketch(faces().group_by(Axis.Z)[0]) as lever_hole:
    Rectangle(lever_width + lever_tolerance / 2, lever_thickness + lever_tolerance / 2)
  extrude(amount=-lever_slot_depth, mode=Mode.SUBTRACT)
  chamfer(edges(Select.LAST).filter_by(Plane.XY).group_by(Axis.Z)[0], lever_chamfer)

slider_knob_body = slider_knob_body.part
slider_knob_body.label = "slider_knob_body"
slider_knob_body.color = Color(0.5, 0.1, 0.1)

slider_knob_insert = slider_knob_insert.part
slider_knob_insert.label = "slider_knob_insert"
slider_knob_insert.color = Color(0.2, 0.2, 0.2)
slider_knob_insert_base_loc = slider_knob_insert.location
slider_knob_insert.location = Location((0, 0, 0), (90, 0, 90))

slider_knob = Compound(
  label="slider_knob",
  children=[
    fast_copy(slider_knob_body),
    copy_located(slider_knob_insert, slider_knob_insert_base_loc),
  ],
)

# show_object(slider_knob_body)
show_object(slider_knob_insert)
show_object(slider_knob)

In [ ]:
export(slider_knob_body)
export(slider_knob_insert)